In [ ]:
# ======================
# CONFIGURATION DU CHEMIN DU PROJET
# ======================
import sys
import os

# Remonter à la racine du projet (depuis notebooks/training/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..','..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Racine du projet ajoutée au PYTHONPATH : {project_root}")
print(f"   sys.path : {sys.path[0]}")

✅ Racine du projet ajoutée au PYTHONPATH : c:\Users\SOL\Downloads\sentiment_analysis_project
   sys.path : c:\Users\SOL\Downloads\sentiment_analysis_project


## importation de model et base de donnée

In [ ]:
from src.preprocessing import preprocess_full_dataset
from src.dataset import create_dataloaders
from src.models import SentimentCNN

# Exécuter le prétraitement
df_clean, recommended_length = preprocess_full_dataset(r'C:\Users\SOL\Downloads\sentiment_analysis_project\data\sentiment_dataset.csv')
# Créer les dataloaders PyTorch
train_loader, test_loader, vocab, train_df, test_df = create_dataloaders(
    df_clean,
    max_length=60,
    batch_size=64
)

# Initialisation du modèle
model = SentimentCNN(
    vocab_size=len(vocab),
    embedding_dim=256,
    n_filters=128,
    filter_sizes=[3, 4, 5],  # Capture les n-grammes de tailles 3, 4 et 5
    output_dim=3,
    dropout=0.5
)

print(f"🧠 Modèle CNN 1D initialisé :")
print(f"   - Embedding dim : 256")
print(f"   - Filtres       : 128 par taille")
print(f"   - Tailles       : {3}, {4}, {5} → capture les n-grammes critiques")
print(f"   - Paramètres    : {sum(p.numel() for p in model.parameters()):,}")

🚀 PRÉTRAITEMENT DU DATASET

✅ Dataset chargé : 31,232 échantillons
🧹 Prétraitement en cours...
✅ 31,202 échantillons conservés

📏 Longueur recommandée (95e percentile) : 56 tokens

📚 Vocabulaire construit : 11703 mots uniques

📦 Dataloaders créés :
   - Train : 391 batches
   - Test  : 98 batches
🧠 Modèle CNN 1D initialisé :
   - Embedding dim : 256
   - Filtres       : 128 par taille
   - Tailles       : 3, 4, 5 → capture les n-grammes critiques
   - Paramètres    : 3,390,723


## l'entrainement

In [ ]:
from src.trainer import train_model
import torch
import torch.nn as nn

# Sélection du device (GPU si disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Entraînement avec monitoring complet
history, model_path = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=30,
    patience=4,
    save_dir=r"C:\Users\SOL\Downloads\sentiment_analysis_project\models\trainary",
    model_name="CNN",
    verbose=True,
)



🚀 DÉBUT DE L'ENTRAÎNEMENT : CNN
Epoch  | Train Loss   Train F1   Val Loss     Val F1     Val Acc    | Time   | Status      
--------------------------------------------------------------------------------
     1 | 2.0970       0.3905     0.9500       0.5484     0.5472     | 4     s | ★ BEST      
     2 | 1.0727       0.4571     0.9226       0.5653     0.5677     | 3     s | ★ BEST      
     3 | 1.0023       0.5028     0.9523       0.4558     0.5235     | 3     s | →           
     4 | 0.9607       0.5393     0.8484       0.6154     0.6151     | 3     s | ★ BEST      
     5 | 0.9192       0.5704     0.8336       0.6293     0.6267     | 3     s | ★ BEST      
     6 | 0.8857       0.5941     0.8096       0.6405     0.6440     | 3     s | ★ BEST      
     7 | 0.8545       0.6137     0.7946       0.6556     0.6563     | 3     s | ★ BEST      
     8 | 0.8217       0.6358     0.8007       0.6458     0.6550     | 3     s | →           
     9 | 0.7962       0.6533     0.7927       0.64

## évaluation

In [ ]:
# Évaluation finale avec visualisations publication
import torch
from src.evaluate import evaluate_model
# Sélection du device (GPU si disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
    class_names=['Négatif', 'Neutre', 'Positif'],
    save_dir="reports/figures",
    model_name="CNN"
)


📊 ÉVALUATION DU MODÈLE : CNN

Accuracy      : 0.6783
F1 Macro      : 0.6789
F1 Weighted   : 0.6797
ROC AUC Micro : 0.8429
ROC AUC Macro : 0.8373

Classification Report:
              precision    recall  f1-score   support

     Négatif     0.7424    0.5621    0.6398      1820
      Neutre     0.5771    0.7542    0.6539      2327
     Positif     0.7986    0.6948    0.7431      2094

    accuracy                         0.6783      6241
   macro avg     0.7060    0.6704    0.6789      6241
weighted avg     0.6996    0.6783    0.6797      6241

   → Matrice de confusion sauvegardée : confusion_matrix.png/csv
   → Courbes ROC sauvegardées : roc_curves.png

✅ Évaluation terminée - Résultats sauvegardés dans : reports\figures\CNN
